In [1]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd
from src.data import load_clean
from sklearn.metrics import accuracy_score, f1_score

train, val, test, train_contra = load_clean()

for n, d in [("train", train), ("val", val), ("test", test)]:
    print(n, "sarcasme global :", round(d["sarcasm"].mean(), 3))

print("\n--- par dialecte ---")
print(pd.concat({n: d.groupby("dialect")["sarcasm"].mean().round(3)
                 for n, d in [("train", train), ("val", val), ("test", test)]}, axis=1))

print("\n--- par sentiment ---")
print(pd.concat({n: d.groupby("sentiment")["sarcasm"].mean().round(3)
                 for n, d in [("train", train), ("val", val), ("test", test)]}, axis=1))

y = val["sarcasm"].values
pred0 = np.zeros_like(y)
print("\nJamais sarcastique -> accuracy:", round(accuracy_score(y, pred0), 3),
      "| F1:", round(f1_score(y, pred0, zero_division=0), 3))

retirés (présents dans test): 4 | (présents dans val): 72 | groupes contradictoires: 47 | doublons restants retirés: 287
tailles -> train: 10834, val: 1255, test: 3000
train sarcasme global : 0.172
val sarcasme global : 0.173
test sarcasme global : 0.274

--- par dialecte ---
         train    val   test
dialect                     
egypt    0.351  0.362  0.507
gulf     0.229  0.206  0.177
levant   0.217  0.291  0.298
magreb   0.371  0.000  0.000
msa      0.108  0.107  0.256

--- par sentiment ---
           train    val   test
sentiment                     
NEG        0.419  0.419  0.423
NEU        0.027  0.029  0.142
POS        0.024  0.024  0.010

Jamais sarcastique -> accuracy: 0.827 | F1: 0.0


In [2]:
import time
from sklearn.pipeline import make_union
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve, average_precision_score
from src.utils import log_result

# Effectifs par dialecte (utile pour Q11)
print(pd.concat({n: d["dialect"].value_counts() for n, d in
                 [("train", train), ("val", val), ("test", test)]}, axis=1))

# Baseline TF-IDF (mots 1-2 + caractères 2-5) + régression logistique pondérée
t0 = time.time()
vec = make_union(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
    TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True))
Xtr = vec.fit_transform(train["norm"])
Xva = vec.transform(val["norm"])
clf = LogisticRegression(class_weight="balanced", max_iter=2000)
clf.fit(Xtr, train["sarcasm"])
p = clf.predict_proba(Xva)[:, 1]
dur = time.time() - t0

# Seuil choisi sur val (maximise le F1 sarcastique)
prec, rec, thr = precision_recall_curve(y, p)
f1s = 2 * prec * rec / (prec + rec + 1e-12)
i = f1s[:-1].argmax()
t = thr[i]
res = dict(f1=f1s[i], pr_auc=average_precision_score(y, p),
           macro_f1=f1_score(y, p >= t, average="macro"))
print(f"seuil={t:.3f} | F1 sarcastique={res['f1']:.4f} | "
      f"PR-AUC={res['pr_auc']:.4f} | macro-F1={res['macro_f1']:.4f}")

log_result(dict(model="tfidf_lr_balanced", preprocessing="norm", seed=0,
                threshold=round(t, 4), f1_val=round(res["f1"], 4),
                prauc_val=round(res["pr_auc"], 4), macrof1_val=round(res["macro_f1"], 4),
                f1_test="", prauc_test="", macrof1_test="",
                train_time_s=round(dur, 1), trained_params=clf.coef_.size + 1),
           path="../results/results.csv")

         train  val  test
dialect                  
msa       7421  867  2323
egypt     2319  260   306
levant     548   55    47
gulf       511   68   322
magreb      35    5     2
seuil=0.542 | F1 sarcastique=0.5148 | PR-AUC=0.5206 | macro-F1=0.7060


In [3]:
rules = {
    "contient URL":          val["tweet"].str.contains(r"https?://", regex=True),
    "contient mention":      val["tweet"].str.contains(r"@\w+", regex=True),
    "guillemets (artefact)": val["has_quote"],
    "RT":                    val["tweet"].str.contains(r"\bRT @", regex=True),
}
for name, r in rules.items():
    r = r.values
    print(f"{name:24s} présent: {r.mean():.3f} | "
          f"sarcasme si présent={y[r].mean():.3f}, si absent={y[~r].mean():.3f} | "
          f"F1 (règle 'présent'={f1_score(y, r):.3f}, règle 'absent'={f1_score(y, ~r):.3f})")

contient URL             présent: 0.398 | sarcasme si présent=0.082, si absent=0.233 | F1 (règle 'présent'=0.115, règle 'absent'=0.362)
contient mention         présent: 0.240 | sarcasme si présent=0.203, si absent=0.164 | F1 (règle 'présent'=0.236, règle 'absent'=0.266)
guillemets (artefact)    présent: 0.628 | sarcasme si présent=0.143, si absent=0.223 | F1 (règle 'présent'=0.225, règle 'absent'=0.304)
RT                       présent: 0.120 | sarcasme si présent=0.199, si absent=0.169 | F1 (règle 'présent'=0.163, règle 'absent'=0.283)


In [4]:
import re, emoji, time
EMOJIS = emoji.UNICODE_EMOJI["en"]

def rm_emoji(s):   return "".join(c for c in s if c not in EMOJIS)
def rm_laugh(s):   return re.sub(r"[هھ]{3,}", " ", s)
def rm_punct(s):   return re.sub(r"[!?؟.]{2,}", ".", s)
def rm_elong(s):   return re.sub(r"(.)\1{2,}", r"\1", s)
def rm_url_mention(s): return re.sub(r"\burl\b|@user", " ", s)

variants = {
    "raw (avec artefact)":     lambda d: d["tweet"],
    "norm (référence)":        lambda d: d["norm"],
    "norm sans URL/mention":   lambda d: d["norm"].apply(rm_url_mention),
    "norm sans emojis":        lambda d: d["norm"].apply(rm_emoji),
    "norm sans rires":         lambda d: d["norm"].apply(rm_laugh),
    "norm sans ponct. rép.":   lambda d: d["norm"].apply(rm_punct),
    "norm sans allongements":  lambda d: d["norm"].apply(rm_elong),
    "norm sans indices (tous)": lambda d: d["norm"].apply(
        lambda s: rm_elong(rm_punct(rm_laugh(rm_emoji(s))))),
}

def run(texts_tr, texts_va):
    vec = make_union(
        TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True))
    Xtr, Xva = vec.fit_transform(texts_tr), vec.transform(texts_va)
    clf = LogisticRegression(class_weight="balanced", max_iter=2000).fit(Xtr, train["sarcasm"])
    p = clf.predict_proba(Xva)[:, 1]
    prec, rec, thr = precision_recall_curve(y, p)
    f1s = 2 * prec * rec / (prec + rec + 1e-12)
    i = f1s[:-1].argmax()
    return f1s[i], average_precision_score(y, p), thr[i]

rows = []
for name, fn in variants.items():
    f1, ap, t = run(fn(train), fn(val))
    rows.append(dict(variante=name, f1_val=round(f1, 4), prauc_val=round(ap, 4), seuil=round(t, 3)))
    print(rows[-1])
pd.DataFrame(rows)

{'variante': 'raw (avec artefact)', 'f1_val': np.float64(0.5047), 'prauc_val': 0.5105, 'seuil': np.float64(0.466)}
{'variante': 'norm (référence)', 'f1_val': np.float64(0.5148), 'prauc_val': 0.5206, 'seuil': np.float64(0.542)}
{'variante': 'norm sans URL/mention', 'f1_val': np.float64(0.511), 'prauc_val': 0.5147, 'seuil': np.float64(0.519)}
{'variante': 'norm sans emojis', 'f1_val': np.float64(0.5084), 'prauc_val': 0.5131, 'seuil': np.float64(0.559)}
{'variante': 'norm sans rires', 'f1_val': np.float64(0.509), 'prauc_val': 0.5152, 'seuil': np.float64(0.537)}
{'variante': 'norm sans ponct. rép.', 'f1_val': np.float64(0.5126), 'prauc_val': 0.5195, 'seuil': np.float64(0.546)}
{'variante': 'norm sans allongements', 'f1_val': np.float64(0.5104), 'prauc_val': 0.5108, 'seuil': np.float64(0.545)}
{'variante': 'norm sans indices (tous)', 'f1_val': np.float64(0.5011), 'prauc_val': 0.5032, 'seuil': np.float64(0.547)}


,variante,f1_val,prauc_val,seuil
0,raw (avec artefact),0.5047,0.5105,0.466
1,norm (référence),0.5148,0.5206,0.542
2,norm sans URL/mention,0.5110,0.5147,0.519
3,norm sans emojis,0.5084,0.5131,0.559
4,norm sans rires,0.5090,0.5152,0.537
5,norm sans ponct. rép.,0.5126,0.5195,0.546
6,norm sans allongements,0.5104,0.5108,0.545
7,norm sans indices (tous),0.5011,0.5032,0.547
